https://www.openintro.org/data/index.php?data=loans_full_schema source for data 

In [1]:
import pandas as pd 
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.dates as mdates
from datetime import datetime
from scipy.stats import linregress
from sklearn.model_selection import train_test_split
import plotly.graph_objects as go
import plotly.io as pio
from sklearn.ensemble import RandomForestRegressor
pio.renderers.default = 'notebook'

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, precision_score

In [2]:
df = pd.read_csv('loans_full_schema.csv')
names = df.columns 
print(df['loan_status'].value_counts())

# Drop columns
df.drop(columns=["verified_income", "verification_income_joint"], inplace=True)
# Show original statuses
print(df["loan_status"].unique())

# Define statuses to KEEP
#good_statuses = ["Fully Paid"]
#bad_statuses = ["Charged Off", "Late (31-120 days)", "Late (16-30 days)", "In Grace Period"] #Current

good_statuses = ["In Grace Period"]
bad_statuses = ["Late (31-120 days)"]
# Anything thats outside the good, and bad shall be dropped. 
df = df[df["loan_status"].isin(good_statuses + bad_statuses)]

# Convert to binary target
df["loan_status"] = df["loan_status"].isin(good_statuses).astype(int)

df

loan_status
Current               9375
Fully Paid             447
In Grace Period         67
Late (31-120 days)      66
Late (16-30 days)       38
Charged Off              7
Name: count, dtype: int64
['Current' 'Fully Paid' 'In Grace Period' 'Late (31-120 days)'
 'Charged Off' 'Late (16-30 days)']


,emp_title,emp_length,state,homeownership,annual_income,debt_to_income,annual_income_joint,debt_to_income_joint,delinq_2y,months_since_last_delinq,...,sub_grade,issue_month,loan_status,initial_listing_status,disbursement_method,balance,paid_total,paid_principal,paid_interest,paid_late_fees
37,supplies clerk,10.0,NJ,MORTGAGE,70000.0,17.06,105000.0,13.27,0,102.0,...,C4,Mar-2018,1,whole,Cash,23455.27,1102.83,544.73,558.10,0.00
121,qc lead,4.0,CA,MORTGAGE,63000.0,15.30,NaN,NaN,0,78.0,...,C5,Feb-2018,1,whole,Cash,9336.71,1032.76,663.29,369.47,0.00
224,regional manager,2.0,NY,RENT,84000.0,7.57,NaN,NaN,0,NaN,...,B5,Jan-2018,0,whole,Cash,33701.09,2311.83,1298.91,1012.92,0.00
283,lead driver,10.0,GA,RENT,85000.0,21.22,NaN,NaN,0,NaN,...,D3,Feb-2018,0,whole,Cash,23760.26,602.25,239.74,362.51,0.00
350,chef,0.0,TX,RENT,14400.0,3.83,NaN,NaN,0,NaN,...,C4,Feb-2018,0,whole,Cash,4889.26,169.27,110.74,58.53,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9602,driver,5.0,GA,MORTGAGE,700000.0,1.48,NaN,NaN,1,23.0,...,C1,Jan-2018,1,fractional,Cash,5078.82,757.83,521.18,221.65,15.00
9629,sqd leader,10.0,TX,RENT,78000.0,14.38,NaN,NaN,0,75.0,...,D2,Feb-2018,0,whole,Cash,14688.62,744.25,311.38,413.80,19.07
9954,medical assistant,4.0,CA,RENT,35000.0,24.70,70000.0,23.97,0,NaN,...,D4,Jan-2018,1,whole,Cash,19201.65,2205.83,798.35,1354.50,52.98
9981,project manager,7.0,KY,MORTGAGE,64000.0,36.49,194000.0,23.72,0,NaN,...,B2,Jan-2018,1,whole,Cash,27089.00,3892.94,2911.00,981.94,0.00


In [3]:
names

Index(['emp_title', 'emp_length', 'state', 'homeownership', 'annual_income',
       'verified_income', 'debt_to_income', 'annual_income_joint',
       'verification_income_joint', 'debt_to_income_joint', 'delinq_2y',
       'months_since_last_delinq', 'earliest_credit_line',
       'inquiries_last_12m', 'total_credit_lines', 'open_credit_lines',
       'total_credit_limit', 'total_credit_utilized',
       'num_collections_last_12m', 'num_historical_failed_to_pay',
       'months_since_90d_late', 'current_accounts_delinq',
       'total_collection_amount_ever', 'current_installment_accounts',
       'accounts_opened_24m', 'months_since_last_credit_inquiry',
       'num_satisfactory_accounts', 'num_accounts_120d_past_due',
       'num_accounts_30d_past_due', 'num_active_debit_accounts',
       'total_debit_limit', 'num_total_cc_accounts', 'num_open_cc_accounts',
       'num_cc_carrying_balance', 'num_mort_accounts',
       'account_never_delinq_percent', 'tax_liens', 'public_record_bankr

In [4]:
def get_value_counts(df_subset, col_name):
    return df_subset[col_name].value_counts()

good_loans = df[df["loan_status"] == 1]
bad_loans = df[df["loan_status"] == 0]

probe_good = get_value_counts(good_loans, "homeownership")
probe_bad = get_value_counts(bad_loans, "homeownership")
print("Shape Good Loans", good_loans.shape)
print(probe_good)

print("Shape Bad Loans", bad_loans.shape)
print("Not fully paid:", probe_bad)


Shape Good Loans (67, 53)
homeownership
MORTGAGE    28
RENT        27
OWN         12
Name: count, dtype: int64
Shape Bad Loans (66, 53)
Not fully paid: homeownership
RENT        33
MORTGAGE    20
OWN         13
Name: count, dtype: int64


# Observation homeownership
Seems that the most common homeownership for "bad lenders" are renting whereas "good lenders" tend to have a mortgage.  

In [5]:
Names = df.columns.drop(["loan_status","emp_title","state","sub_grade"]).tolist()
print(Names)


['emp_length', 'homeownership', 'annual_income', 'debt_to_income', 'annual_income_joint', 'debt_to_income_joint', 'delinq_2y', 'months_since_last_delinq', 'earliest_credit_line', 'inquiries_last_12m', 'total_credit_lines', 'open_credit_lines', 'total_credit_limit', 'total_credit_utilized', 'num_collections_last_12m', 'num_historical_failed_to_pay', 'months_since_90d_late', 'current_accounts_delinq', 'total_collection_amount_ever', 'current_installment_accounts', 'accounts_opened_24m', 'months_since_last_credit_inquiry', 'num_satisfactory_accounts', 'num_accounts_120d_past_due', 'num_accounts_30d_past_due', 'num_active_debit_accounts', 'total_debit_limit', 'num_total_cc_accounts', 'num_open_cc_accounts', 'num_cc_carrying_balance', 'num_mort_accounts', 'account_never_delinq_percent', 'tax_liens', 'public_record_bankrupt', 'loan_purpose', 'application_type', 'loan_amount', 'term', 'interest_rate', 'installment', 'grade', 'issue_month', 'initial_listing_status', 'disbursement_method', 'bal

In [6]:
# "Names" contains each col name. 

# Remove each row which contains NaN and its between 1-9. 
col_to_remove = []

for names in Names: 
    number_of_nan = df[names].isna().sum()
    if (number_of_nan > 0): 
        print(names, df[names].isna().sum())
    if (number_of_nan > 0 and number_of_nan < 10): 
        col_to_remove.append(names)
    
for col in col_to_remove: # Remove rows which contain NaN and the number of NaN is not significant 
    df = df.dropna(subset=[col])       

# Remove annual_income_joint, debt_to_income_joint (Many NaN values, which deals with having a partner) 
# Remove columns which has text values (such as homeownership = rent). 
Names_to_exclude = []
Names_to_exclude.append("annual_income_joint")
Names_to_exclude.append("debt_to_income_joint") 
text_cols = df.select_dtypes(include="object").columns.tolist()


X_without_text_cols = [] 
for col in Names: 
    if col not in text_cols and col not in Names_to_exclude: # Exclude text values, as well as specific 
        X_without_text_cols.append(col)

# These values deal with time, thus NaN are for these columns treated as 0. 
cols_to_zero = [ 
    "emp_length",
    "months_since_last_delinq",
    "months_since_90d_late",
    "months_since_last_credit_inquiry",
    "num_accounts_120d_past_due", 
    "debt_to_income"
]

df[cols_to_zero] = df[cols_to_zero].fillna(0)

# Update names & remove constant values 
X_names_wihtout_constant_remove = []
for names in Names: 
    if df[names].nunique() == 1: 
        print(f"{names} is constant") 
        df = df.drop(columns=[names])  
        X_names_wihtout_constant_remove.append(names)
        
X_to_train = []
for col in X_without_text_cols: 
    if col not in X_names_wihtout_constant_remove: 
        X_to_train.append(col)
X_without_text_cols = X_to_train 

emp_length 11
debt_to_income 1
annual_income_joint 104
debt_to_income_joint 104
months_since_last_delinq 79
months_since_90d_late 94
months_since_last_credit_inquiry 10
num_accounts_120d_past_due 15
current_accounts_delinq is constant
num_accounts_120d_past_due is constant
num_accounts_30d_past_due is constant


C:\Users\Patri\AppData\Local\Temp\ipykernel_11968\4210298128.py:39: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [7]:
X_values_Names = X_without_text_cols 
# We want to model based on loans.
Y_values_Names = 'loan_status'

# Define X and y
X = df[X_without_text_cols]
y = df[Y_values_Names]

# Split into training and testing sets (80% train, 20% test)
#X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

from imblearn.over_sampling import SMOTE 
sm = SMOTE(random_state=42) 
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)

# 1. Fit the model
model = LogisticRegression(max_iter=1000)
model.fit(X_train_res, y_train_res.values.ravel())  # .ravel() flattens y if it's a DataFrame

# 2. Predict on test set
y_pred = model.predict(X_test) #Have or not have? 

# 3. Get predicted probabilities
y_prob = model.predict_proba(X_test)[:, 1]  # Probability of class 1 (diabetic) => Actual prob

# 4. Evaluate performance
print("Accuracy:", accuracy_score(y_test, y_pred), "Precission:", precision_score(y_test, y_pred))

feature_importance = pd.DataFrame({
    'Feature': X_values_Names,
    'Coefficient': model.coef_[0]
})

print(feature_importance)

from sklearn.metrics import classification_report 
print(classification_report(y_test, y_pred))

print(df[Y_values_Names].value_counts())

Accuracy: 0.7777777777777778 Precission: 0.7333333333333333
                             Feature  Coefficient
0                         emp_length     0.007503
1                      annual_income     0.000003
2                     debt_to_income     0.002903
3                          delinq_2y     0.000612
4           months_since_last_delinq    -0.000876
5               earliest_credit_line    -0.000704
6                 inquiries_last_12m     0.001041
7                 total_credit_lines     0.001917
8                  open_credit_lines     0.002263
9                 total_credit_limit     0.000010
10             total_credit_utilized    -0.000008
11          num_collections_last_12m    -0.000007
12      num_historical_failed_to_pay    -0.000512
13             months_since_90d_late    -0.016810
14      total_collection_amount_ever     0.004211
15      current_installment_accounts    -0.004468
16               accounts_opened_24m    -0.003113
17  months_since_last_credit_inquiry    

C:\Users\Patri\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:470: ConvergenceWarning:

lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression



In [8]:
X_values_Names = X_without_text_cols 
# We want to model based on loans.
Y_values_Names = 'loan_status'

# Define X and y
X = df[X_without_text_cols]
y = df[Y_values_Names]

from sklearn.feature_selection import SelectKBest, f_classif
selector = SelectKBest(f_classif, k=10)
X_new = selector.fit_transform(X, y)

selected_mask = selector.get_support() 
selected_features = X.columns[selected_mask] 


# Split into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X_new, y, test_size=0.2, random_state=42)

from imblearn.over_sampling import SMOTE 
sm = SMOTE(random_state=42) 
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)

# 1. Fit the model
model = LogisticRegression(max_iter=1000, class_weight="balanced")
model.fit(X_train, y_train.values.ravel())  # .ravel() flattens y if it's a DataFrame

# 2. Predict on test set
y_pred = model.predict(X_test) #Have or not have? 

# 3. Get predicted probabilities
y_prob = model.predict_proba(X_test)[:, 1]  # Probability of class 1 (diabetic) => Actual prob

# 4. Evaluate performance
print("Accuracy:", accuracy_score(y_test, y_pred), "Precission:", precision_score(y_test, y_pred))

feature_importance = pd.DataFrame({
    'Feature': selected_features,
    'Coefficient': model.coef_[0]
})

print(feature_importance)

from sklearn.metrics import classification_report 
print(classification_report(y_test, y_pred))


Accuracy: 0.6666666666666666 Precission: 0.6
                     Feature  Coefficient
0             debt_to_income     0.010556
1         total_credit_lines     0.026888
2          open_credit_lines     0.067231
3  num_satisfactory_accounts    -0.178129
4  num_active_debit_accounts    -0.241509
5    num_cc_carrying_balance     0.347260
6          num_mort_accounts     0.273633
7                 paid_total     0.002957
8             paid_principal    -0.001352
9              paid_interest    -0.002258
              precision    recall  f1-score   support

           0       0.68      0.88      0.77        17
           1       0.60      0.30      0.40        10

    accuracy                           0.67        27
   macro avg       0.64      0.59      0.58        27
weighted avg       0.65      0.67      0.63        27



C:\Users\Patri\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:470: ConvergenceWarning:

lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression



In [9]:
np.unique(y_pred, return_counts=True)
X_test.shape
"loan_status" in X.columns


False